# M-Lab Calibration Dashboard

### Warning: This is a pre-release of experimental software.
It is not running on production infrastructure and is subject to changes or deletion without notification.
For general documentation on the methodology see the [README](/d/20251106a/).

Sites with high scores (ratio >> 1.0 or KSdistance >> 0.0)
are probably not calibrated, and should be inspected by other means (e.g. the Regional Details Dashboard)
to evaluate their health.

This dashboard presents a fast way to evaluate the most important MLab site performance metrics,
but does not provide complete coverage of all possible calibration problems.

See the [full documentation](https://docs.google.com/document/d/1t2oxmFSxNae6B89mR7whhaEn1ye1sIJi8CduKOlFDGc/edit?tab=t.0)
for a discussion of the limitations of this test.

In [ ]:
# --- Setup ---
import os, sys, json
from datetime import datetime, date, time, timedelta, timezone

# Locate the repo root (directory containing `converter/`) regardless of where
# Voila/Jupyter is launched from.
_root = os.path.abspath(os.getcwd())
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, "converter")):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

import ipywidgets as widgets
import plotly.graph_objects as go
import pandas as pd
from IPython.display import display, HTML, Markdown

from converter import query_builder as qb, runtime as rt
from converter.widget_builder import Controls

client = rt.bq_client()

# Dashboard variable metadata baked in at conversion time.
VARIABLES = json.loads(r"""
[
  {
    "name": "field",
    "type": "custom",
    "label": "",
    "description": "Select the 4th data column.  (Linear fields don't display correctly.) \nMethod must be set to experimental FIRST.",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "MeanThroughputMbps", "value": "MeanThroughputMbps" },
      { "text": "uploadMeanThroughputMbps", "value": "uploadMeanThroughputMbps" },
      { "text": "LossRate", "value": "LossRate" }
    ],
    "current": { "value": "MeanThroughputMbps" },
    "query_sql": "MeanThroughputMbps, uploadMeanThroughputMbps, LossRate"
  },
  {
    "name": "region",
    "type": "query",
    "label": "Region",
    "description": "Used to select Client ISPs to be evaluated in  region. ",
    "hide": 0,
    "multi": true,
    "options": [],
    "current": {
      "value": [
        "$__all"
      ]
    },
    "query_sql": "# From: 2024-12-20 Prototype cached site picker  \n\nSELECT\n  CONCAT(metro, ' ', ContinentCode, ' ', City, ', ', CountryCode) AS text,\n  metro AS value,\nFROM (\n  SELECT\n    REGEXP_EXTRACT(site, '^([a-z]{3})') AS metro,\n    ANY_VALUE(ContinentCode) AS ContinentCode,\n    ANY_VALUE(City) AS City,\n    ANY_VALUE(CountryCode) AS CountryCode,\n    -- Uses ${datasource} implicitly\n  FROM `mlab-collaboration.mm_preproduction.cached_metadata`\n  GROUP BY metro\n  ORDER BY  ContinentCode, CountryCode, metro\n)\n",
    "default_select": "all"
  },
  {
    "name": "radius",
    "type": "custom",
    "label": "Radius (kM)",
    "description": "Radius from the anchor metro for selecting servers",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "1", "value": "1" },
      { "text": "100", "value": "100" },
      { "text": "500", "value": "500" }
    ],
    "current": { "value": "100" },
    "query_sql": "1, 100, 500"
  },
  {
    "name": "ISPcount",
    "type": "custom",
    "label": "",
    "description": "Number of ISPs to include when searching for calibration triplets",
    "hide": 0,
    "multi": false,
    "options": [
      { "text": "5", "value": "5" },
      { "text": "2", "value": "2" },
      { "text": "10", "value": "10" }
    ],
    "current": { "value": "5" },
    "query_sql": "5, 2, 10"
  }
]
""")


In [ ]:
# --- URL parameter presets (webapp mode) ---
# Voila injects the request query string into os.environ["QUERY_STRING"] before
# executing the notebook.  get_query_string() also handles the preheat-kernel
# case (blocks until the request arrives).  Falls back gracefully in plain
# Jupyter where neither is set.
# For scripted or test overrides, set DASH_PRESETS to a JSON object.
import urllib.parse

url_params = {}
try:
    from voila.utils import get_query_string
    _qs = get_query_string() or ""
    for _k, _vs in urllib.parse.parse_qs(_qs).items():
        url_params[_k] = _vs[0] if len(_vs) == 1 else _vs
except Exception:
    pass

_env = os.environ.get("DASH_PRESETS")
if _env:
    url_params.update(json.loads(_env))


In [ ]:
# --- Dashboard controls (dropdowns; query-backed ones are chained) ---
ctrl = Controls(VARIABLES, client, presets=url_params)
w_run = widgets.Button(description="Run / Refresh", button_style="primary", icon="play")

# Date range for experimental/live backends — ignored when method=cached.
# Defaults to one week ending on the most recent Sunday (UTC).
_today_utc  = datetime.now(timezone.utc).date()
_days_back  = (_today_utc.weekday() + 1) % 7   # 0 on Sunday, 1 on Monday …
_end_date   = _today_utc - timedelta(days=_days_back)
_start_date = _end_date  - timedelta(days=6)
w_to   = widgets.DatePicker(value=_end_date,   description='End (UTC)',
                             style={"description_width": "90px"})
w_from = widgets.DatePicker(value=_start_date, description='Start (UTC)',
                             style={"description_width": "90px"})
_date_label = widgets.HTML('')   # filled from query results after Run


In [ ]:
# --- Calibration panels ---
_DATASET  = "mlab-collaboration.mm_preproduction"
_METHOD   = "cached"   # expose in a later pass if live/exp needed
_X_AXIS   = "none"
_BIN_SIZE = 50

out = widgets.Output()


def _diagnostics(ctx):
    rows = [(k, ", ".join(v) if isinstance(v, list) else str(v))
            for k, v in ctx.items()]
    return pd.DataFrame(rows, columns=["variable", "value"])


def render(_=None):
    ctx = ctrl.context()
    to_dt   = (datetime.combine(w_to.value,   time(), tzinfo=timezone.utc)
               if w_to   and w_to.value   else datetime.now(timezone.utc))
    from_dt = (datetime.combine(w_from.value, time(), tzinfo=timezone.utc)
               if w_from and w_from.value else to_dt - timedelta(days=7))

    region_regex = qb.format_regex(ctx.get("region") or [])

    out.clear_output(wait=True)
    with out:
        try:
            df = rt.run_calibration_report(
                client, _METHOD, _X_AXIS, _BIN_SIZE,
                ctx.get("field", "MeanThroughputMbps"),
                from_dt, to_dt,
                region_regex,
                int(ctx.get("radius", 100)),
                int(ctx.get("ISPcount", 5)),
                _DATASET,
            )
        except Exception as exc:
            display(HTML(f"<pre>query failed: {exc}</pre>"))
            df = None

        if df is not None and not df.empty:
            display(Markdown("### Scatter plot of KSdistance and ratio"))
            _ratio_col = "Ratio" if "Ratio" in df.columns else "ratio"
            _scatter_df = df[df[_ratio_col] >= 1.0].copy()
            display(go.FigureWidget(rt.plotly_calibration_scatter(_scatter_df)))

            display(Markdown("### Calibration report"))
            _table_df = df.drop(columns=["BCargs", "Breadcrumb"], errors="ignore")
            display(HTML(
                '<div style="height:500px;overflow:auto">'
                + _table_df.to_html(index=False, na_rep="")
                + '</div>'
            ))

        _diag_out = widgets.Output()
        with _diag_out:
            display(_diagnostics(ctx))
        _diag_acc = widgets.Accordion(children=[_diag_out])
        _diag_acc.set_title(0, "Selector Diagnostics")
        _diag_acc.selected_index = None
        display(_diag_acc)


w_run.on_click(render)
if url_params:
    render()


In [ ]:
# --- Display the app ---
_date_row = (widgets.HBox([w_from, w_to],
                          layout=widgets.Layout(margin='2px 0'))
             if w_from is not None else widgets.HTML(''))
display(widgets.VBox([ctrl.box, _date_label, _date_row, w_run, out]))
